In [1]:
%load_ext autoreload
%autoreload 2

# Entrenamiento del modelo

## Aprendizaje supervisado (SL)

Con el dataset listo, procederemos a configurar y entrenar nuestra red neuronal. Este proceso se divide en la carga de datos, la definición de la arquitectura y la ejecución del ciclo de entrenamiento supervisado.

### Carga del dataset

Primero, importamos los datos preparados. En este paso, transformamos el archivo `.data` en un objeto manejable por el framework para el entrenamiento.

In [ ]:
from preprocessing.dataset import load_dataset

# Cargamos el dataset que balanceamos en pasos anteriores
filename = "dataset_ejemplo_uc.data"
data = load_dataset(filename)

Dataset dataset_ejemplo_uc.data cargado con 1000 muestras.


### Configuración de la arquitectura

Definiremos los hiperparámetros del **CPMPTransformer**. Es fundamental que las dimensiones de entrada (`H_dim`, `C_dim`, `X_dim`) coincidan con el formato del **Input Adapter** utilizado durante la generación de los datos.

In [ ]:
from models.actions.cpmp_transformer_V2 import CPMPTransformer

# Parámetros de la arquitectura
model = CPMPTransformer(
    H_dim=12,                # Dimensión de altura (máx stacks)
    C_dim=3,                 # Features por contenedor
    X_dim=3,                 # Features globales del stack
    d_model=32,              # Dimensión interna del Transformer
    nhead=4,                 # Número de cabezales de atención
    num_layers=4,            # Número de capas del encoder
    ff_dim_multiplier=2,     # Multiplicador de la capa Feed-Forward
    dropout=0.3              # Regularización para evitar overfitting
)

### Ejecución del entrenamiento

Utilizaremos la función `sl_train`. Esta se encarga de dividir los datos en conjuntos de entrenamiento y validación, gestionar el optimizador y aplicar **Early Stopping** mediante el parámetro `patience`.

In [9]:
from training.training import sl_train, Accuracy, CrossEntropyLoss

# Configuración del ciclo de entrenamiento
epochs = 100
train_size = 800
test_size = 200
batch_size = 8
learning_rate = 1e-4

# Definimos la función de pérdida y las métricas de evaluación
loss_function = CrossEntropyLoss()
metrics = [Accuracy()]

# Iniciamos el proceso de entrenamiento
model = sl_train(
    model, epochs, data, train_size, test_size, batch_size, 
    learning_rate, weight_decay=1e-4, 
    loss_functions=[loss_function], 
    patience=10, 
    metrics=[metrics], 
    seed=42
)

ℹ️ Usando dispositivo: cpu

Epoch 1/100
    Train CrossEntropy: 2.6846
    Val CrossEntropy: 2.6434
 | Accuracy: 18.00%
Epoch 2/100
    Train CrossEntropy: 2.6796
    Val CrossEntropy: 2.6387
 | Accuracy: 24.00%
Epoch 3/100
    Train CrossEntropy: 2.6724
    Val CrossEntropy: 2.6331
 | Accuracy: 24.50%
Epoch 4/100
    Train CrossEntropy: 2.6633
    Val CrossEntropy: 2.6207
 | Accuracy: 24.00%
Epoch 5/100
    Train CrossEntropy: 2.6493
    Val CrossEntropy: 2.6099
 | Accuracy: 21.50%
Epoch 6/100
    Train CrossEntropy: 2.6394
    Val CrossEntropy: 2.5899
 | Accuracy: 22.00%
Epoch 7/100
    Train CrossEntropy: 2.6405
    Val CrossEntropy: 2.5723
 | Accuracy: 21.00%
Epoch 8/100
    Train CrossEntropy: 2.6252
    Val CrossEntropy: 2.5701
 | Accuracy: 20.50%
Epoch 9/100
    Train CrossEntropy: 2.6154
    Val CrossEntropy: 2.5690
 | Accuracy: 19.00%
Epoch 10/100
    Train CrossEntropy: 2.6132
    Val CrossEntropy: 2.5545
 | Accuracy: 20.00%
Epoch 11/100
    Train CrossEntropy: 2.5999
    Val

### Persistencia del modelo

Una vez finalizado el entrenamiento, guardaremos el modelo. Esto nos permitirá cargarlo posteriormente en la parte de **Validación** para resolver instancias de benchmark.

In [10]:
from training.training import save_model

# Guardamos los pesos y la configuración del modelo bajo un nombre identificativo
save_model(model, "ejemplo_transformer_sl")

✅ Modelo guardado en /home/oscar/Escritorio/CPMP-Framework/models/ejemplo_transformer_sl.pth


##  Self-Supervised Learning (Auto-Entrenamiento)

Una vez que disponemos de un modelo entrenado y validado, es posible que busquemos seguir refinando sus resultados.

Cuando el modelo logra superar el desempeño de la heurística clásica con la que fue entrenado inicialmente, ya no es eficiente seguir utilizando esa misma heurística para generar nuevos datos. En este escenario, implementaremos un enfoque donde **el mismo modelo se utiliza como solver para generar su propio dataset de entrenamiento**, permitiendo un ciclo de mejora continua.

### Carga del modelo base

Comenzaremos cargando los pesos del modelo que entrenamos previamente mediante aprendizaje supervisado.

In [ ]:
from training.training import load_model
from models.actions.cpmp_transformer_V2 import CPMPTransformer

# Cargamos el modelo preentrenado que servirá como base
model = load_model(CPMPTransformer, "ejemplo_transformer_sl")

### Configuración y ejecución del ciclo de entrenamiento

Para este proceso, emplearemos la función `rl_train`. A diferencia del flujo anterior, utilizaremos el objeto `DataGenerationConfigRL` para especificar qué conjunto de instancias debe resolver el modelo en cada iteración para construir su nuevo dataset.

In [6]:
from training.training import rl_train, CrossEntropyLoss, Accuracy, DataGenerationConfigRL
from generation.adapters.input import EnrichedLayoutAdapter, Layout4DAdapterV2
from generation.adapters.input.stack_features import StackFeaturesAdapterV1
from generation.adapters.output import ActionAdapter

# Definimos las funciones de pérdida y las métricas
loss_functions = [CrossEntropyLoss()]
metrics = [[Accuracy()]]

# Configuramos la generación de datos en tiempo de entrenamiento
datagen_config = DataGenerationConfigRL(
    instance_sets=["ejemplo_uc"],    # Carpetas con las instancias a resolver
    H=[7],                           # Altura de las instancias
    max_steps=50,                    # Límite de pasos para evitar bucles infinitos
    # Formato del adaptador de entrada: (Clase, Componente1, Componente2, S_max, H_max)
    input_adapter_config=(EnrichedLayoutAdapter, Layout4DAdapterV2, StackFeaturesAdapterV1, 10, 8),
    output_adapter_config=(ActionAdapter, ),
    num_workers=None                 # Paralelismo automático
)

# Ejecutamos el entrenamiento por iteraciones
# iterations=2 indica cuántas veces el modelo regenerará el dataset completo
model = rl_train(
    model, iterations=2, datagen_config=datagen_config, 
    epochs=10, train_size=800, test_size=200, batch_size=16, 
    learning_rate=1e-4, weight_decay=1e-4, loss_functions=loss_functions, 
    patience=5, metrics=metrics, seed=42
)

ℹ️ Usando dispositivo: cpu
Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/tmp_train.data (Tamaño 317)
Datos guardados en: /home/oscar/Escritorio/CPMP-Framework/data/tmp_test.data (Tamaño 83)
Tamaño datasets | Train: 317 | Test: 83
Costo promedio | Train: 24.03 | Test: 25.93


/home/oscar/Escritorio/CPMP-Framework/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Epoch 1/10
    Train CrossEntropy: 2.5433
    Val CrossEntropy: 2.5664
 | Accuracy: 14.46%
Epoch 2/10
    Train CrossEntropy: 2.5461
    Val CrossEntropy: 2.5679
 | Accuracy: 13.25%
Epoch 3/10
    Train CrossEntropy: 2.5575
    Val CrossEntropy: 2.5628
 | Accuracy: 13.25%
Epoch 4/10
    Train CrossEntropy: 2.5416
    Val CrossEntropy: 2.5673
 | Accuracy: 12.05%
Epoch 5/10
    Train CrossEntropy: 2.5299
    Val CrossEntropy: 2.5695
 | Accuracy: 10.84%
Epoch 6/10
    Train CrossEntropy: 2.5338
    Val CrossEntropy: 2.5620
 | Accuracy: 10.84%
Epoch 7/10
    Train CrossEntropy: 2.5336
    Val CrossEntropy: 2.5564
 | Accuracy: 10.84%
Epoch 8/10
    Train CrossEntropy: 2.5125
    Val CrossEntropy: 2.5686
 | Accuracy: 10.84%
Epoch 9/10
    Train CrossEntropy: 2.5082
    Val CrossEntropy: 2.5753
 | Accuracy: 12.05%
Epoch 10/10
    Train CrossEntropy: 2.5122
    Val CrossEntropy: 2.5572
 | Accuracy: 10.84%
Mejor modelo (CrossEntropy): 2.5564 (Epoch 7)

Datos guardados en: /home/oscar/Escritori